In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
# STEP 1: DOWNLOAD AND LOAD THE DATASET
# ==========================================
print("📥 Downloading latest version of the dataset...")
path = kagglehub.dataset_download("ishanjha100/student-passfail-data")

csv_files = glob.glob(os.path.join(path, "*.csv"))
if not csv_files:
    raise FileNotFoundError("Could not find any CSV file in the downloaded folder.")

csv_path = csv_files[0]
df = pd.read_csv(csv_path)

# Drop the 'student_id' column as it has zero predictive value
if 'student_id' in df.columns:
    df = df.drop(columns=['student_id'])

📥 Downloading latest version of the dataset...


In [3]:
# STEP 3: SEPARATE FEATURES AND TARGET
# ==========================================
X = df.drop(columns=['pass']) 
y = df['pass']

In [4]:
# STEP 4: TRAIN-TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [5]:
# STEP 5: FEATURE SCALING
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# STEP 6: TRAIN LOGISTIC REGRESSION (WITH BALANCING)
# ==========================================
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)
print("✅ Model training and optimization complete.")

✅ Model training and optimization complete.


In [7]:
# ==========================================
# STEP 9: NEW INTERACTIVE INFERENCE ENGINE
# ==========================================
print("\n=======================================================")
print("🎯 WELCOME TO THE STUDENT PASS/FAIL PREDICTION SYSTEM 🎯")
print("=======================================================\n")

while True:
    print("Enter the metrics for a new student (or type 'exit' to quit):")
    
    try:
        # 1. Gather raw inputs from console
        attendance_input = input("👉 Enter Attendance Percentage (0-100): ")
        if attendance_input.lower() == 'exit':
            break
        
        homework_input = input("👉 Enter Homework Completion Percentage (0-100): ")
        midterm_input = input("👉 Enter Midterm Score: ")
        study_hours_input = input("👉 Enter Weekly Study Hours: ")
        
        # Convert values to float for processing
        attendance = float(attendance_input)
        homework = float(homework_input)
        midterm = float(midterm_input)
        study_hours = float(study_hours_input)
        
        # 2. Structure raw input data into a clean 2D DataFrame
        # IMPORTANT: Column names must perfectly match original dataset 'X' columns!
        raw_custom_data = pd.DataFrame([{
            'attendance_pct': attendance,
            'homework_pct': homework,
            'midterm_score': midterm,
            'study_hours_per_week': study_hours
        }])
        
        # 3. Transform data using the ALREADY FITTED training scaler
        # Crucial conceptual point: Use .transform(), NEVER .fit_transform() here!
        scaled_custom_data = scaler.transform(raw_custom_data)
        
        # 4. Perform inference predictions
        prediction = model.predict(scaled_custom_data)[0]
        probabilities = model.predict_proba(scaled_custom_data)[0] # Gives internal confidence matrix
        
        # 5. Render user-facing predictions output
        print("\n" + "="*40)
        print("🧠 AI INTERFERENCE ENGINE RESULTS:")
        print("="*40)
        
        if prediction == 1:
            print(f"🟢 STATUS: PASS")
            print(f"📊 Confidence Level: {probabilities[1] * 100:.2f}% chance of passing.")
        else:
            print(f"🔴 STATUS: FAIL")
            print(f"📊 Risk Level: {probabilities[0] * 100:.2f}% chance of failing.")
            
        print("="*40 + "\n")
        
    except ValueError:
        print("\n❌ Error: Please enter valid numbers for scores and percentages.\n")
    except Exception as e:
        print(f"\n❌ An unexpected error occurred: {e}\n")

print("🚪 System closed. Happy Portfolio Building!")


🎯 WELCOME TO THE STUDENT PASS/FAIL PREDICTION SYSTEM 🎯

Enter the metrics for a new student (or type 'exit' to quit):


👉 Enter Attendance Percentage (0-100):  55
👉 Enter Homework Completion Percentage (0-100):  78
👉 Enter Midterm Score:  88
👉 Enter Weekly Study Hours:  14



🧠 AI INTERFERENCE ENGINE RESULTS:
🟢 STATUS: PASS
📊 Confidence Level: 99.45% chance of passing.

Enter the metrics for a new student (or type 'exit' to quit):


👉 Enter Attendance Percentage (0-100):  45
👉 Enter Homework Completion Percentage (0-100):  33
👉 Enter Midterm Score:  23
👉 Enter Weekly Study Hours:  2



🧠 AI INTERFERENCE ENGINE RESULTS:
🔴 STATUS: FAIL
📊 Risk Level: 99.99% chance of failing.

Enter the metrics for a new student (or type 'exit' to quit):


👉 Enter Attendance Percentage (0-100):  exit


🚪 System closed. Happy Portfolio Building!
